# QQQ Technical Indicator Feature Importance — Project Setup

**Purpose:** make the research scope, data contract, target, commands, and deliverables understandable before any analysis code is read. This notebook is a setup guide and environment check; it does not download data or report market findings.

## Goal

1. Describe how QQQ return, volatility, drawdown, volume, and trend change over time.
2. Test whether information known at the close of day *t* predicts whether QQQ closes higher on day *t+1*.
3. Compare a time-aware model with a majority-class baseline on unseen future observations.

> This is an educational research workflow, not investment advice.

## Setup

### Data contract

| Item | Definition |
|---|---|
| Instrument | Invesco QQQ Trust (`QQQ`) |
| Provider | Nasdaq public historical quote endpoint |
| Grain | One row per trading day |
| Target price | Nasdaq daily close |
| Configured period | Read from `config.json` |
| Raw-data rule | Immutable after retrieval; retain retrieval metadata |

Nasdaq historical quotes are used for coursework. Historical revisions can change a later download, so preserve the raw JSON extract used for every submitted result.

### Target and leakage rules

For row *t*: `target_up_next_day = 1` when `Close(t+1) > Close(t)`, otherwise `0`.

- Drop the final row because its next-day outcome is unknown.
- A feature on row *t* may use information available only through the close of *t*.
- Fit scalers, imputers, selectors, and models only on the training period.
- Preserve chronological order; never use a random split for the primary evaluation.
- Compare against the majority class calculated from the training set.

The configured primary metric is balanced accuracy, which gives equal importance to up and down classes. Secondary metrics provide additional context.

In [1]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'config.json').exists():
            return candidate
    raise FileNotFoundError('Could not locate config.json')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pipeline import (
    RAW_DATA_PATH, RAW_METADATA_PATH, PROCESSED_DATA_PATH, load_config,
)

config = load_config()

print(f'Python: {sys.version.split()[0]}')
print(f'Project root: {PROJECT_ROOT}')
print(f"Study: {config['symbol']} from {config['start_date']} to {config['end_date']}")
print('Target: 1 when Close(t+1) > Close(t), otherwise 0')
print(f'Raw response: {RAW_DATA_PATH.relative_to(PROJECT_ROOT)}')
print(f'Raw metadata: {RAW_METADATA_PATH.relative_to(PROJECT_ROOT)}')
print(f'Processed data: {PROCESSED_DATA_PATH.relative_to(PROJECT_ROOT)}')

Python: 3.14.7
Project root: /Users/tien/Development/PhD_SUT_ENG552209_BigDataAnalytics/qqq-market-insight
Study: QQQ from 2021-01-01 to 2026-09-16
Target: 1 when Close(t+1) > Close(t), otherwise 0
Raw response: data/raw/qqq_nasdaq_raw.json
Raw metadata: data/raw/qqq_nasdaq_raw.metadata.json
Processed data: data/processed/qqq_features.csv


## Steps

From the repository root, create and activate an environment, install dependencies, and launch Jupyter:

```bash
python -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
python -m jupyter lab
```

To prove this notebook runs cleanly from top to bottom:

```bash
python -m jupyter nbconvert --execute --to notebook --inplace notebooks/qqq_feature_importance.ipynb
```

Future notebooks should follow the sequence `01_data_quality_and_eda.ipynb`, `02_feature_engineering.ipynb`, and `03_time_series_modeling.ipynb`.

In [2]:
required_directories = [
    'data/raw', 'data/processed', 'outputs/figures', 'outputs/tables',
    'deliverables/report', 'deliverables/presentation', 'notebooks',
    'src', 'tests', 'docs', 'prompts',
]
required_files = ['README.md', 'requirements.txt', 'config.json']
missing_directories = [p for p in required_directories if not (PROJECT_ROOT / p).is_dir()]
missing_files = [p for p in required_files if not (PROJECT_ROOT / p).is_file()]
assert not missing_directories, f'Missing directories: {missing_directories}'
assert not missing_files, f'Missing files: {missing_files}'
required_config_keys = {
    'symbol', 'asset_class', 'start_date', 'end_date',
    'test_fraction', 'random_state', 'permutation_repeats',
}
assert required_config_keys <= config.keys()
assert 0 < config['test_fraction'] < 1
assert config['start_date'] < config['end_date']
print('PASS: structure, required files, and configurable parameters are valid.')

PASS: structure, required files, and configurable parameters are valid.


## Checks

A valid setup must satisfy all of the following:

- The preceding cell prints `PASS`.
- A fresh-kernel **Run All** completes without manual state.
- Parameters are changed in `config.json`, not copied into many notebooks.
- Raw downloads are preserved and their retrieval time is recorded.
- Training, validation, and test observations remain chronological.
- Generated outputs follow the conventions in the README.

## Next Steps

Expected project deliverables are: (1) executed data-quality/EDA and modeling notebooks, (2) a documented processed dataset, (3) final figures and CSV tables, (4) model metadata with features, splits, metrics, and package versions, and (5) a concise report of findings and limitations.

The data-acquisition stage should download QQQ through `src.pipeline`, save the unchanged response as `data/raw/qqq_nasdaq_raw.json`, save request URL, UTC retrieval time, and SHA-256 checksum in `data/raw/qqq_nasdaq_raw.metadata.json`, and refuse to overwrite either artifact unless `--force` is explicitly requested. Parsing, cleaning, and feature engineering will later produce `data/processed/qqq_features.csv`.